In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Hotel Review") \
    .getOrCreate()

In [8]:
df = spark.read.csv("hotel-review.csv", header=True, inferSchema=True, sep=";" )
df.show(5)

+---+-------------------------------------------------------------------------------------+-------------+-----------+--------+
|  1|Vừa qua tôi có dùng dịch vụ tại Khách Sạn TTC Hotel Premium Ngọc Lan Ngọc Lan Đà Lạt.|      GENERAL|      HOTEL|positive|
+---+-------------------------------------------------------------------------------------+-------------+-----------+--------+
|  1|                                                                 Vừa qua tôi có dù...|      GENERAL|    SERVICE|positive|
|  2|                                                                 Tuy nhiên buffet ...|      QUALITY|FOOD&DRINKS|negative|
|  2|                                                                 Tuy nhiên buffet ...|STYLE&OPTIONS|FOOD&DRINKS|negative|
|  3|                                                                 Nhìn chung dịch v...|      QUALITY|FOOD&DRINKS|negative|
|  3|                                                                 Nhìn chung dịch v...|      GENERAL|    SE

In [10]:
print(df.columns)

['1', 'Vừa qua tôi có dùng dịch vụ tại Khách Sạn TTC Hotel Premium Ngọc Lan Ngọc Lan Đà Lạt.', 'GENERAL', 'HOTEL', 'positive']


In [11]:
df = df.toDF("id", "review", "category", "aspect", "sentiment")

df.show(5)

+---+--------------------+-------------+-----------+---------+
| id|              review|     category|     aspect|sentiment|
+---+--------------------+-------------+-----------+---------+
|  1|Vừa qua tôi có dù...|      GENERAL|    SERVICE| positive|
|  2|Tuy nhiên buffet ...|      QUALITY|FOOD&DRINKS| negative|
|  2|Tuy nhiên buffet ...|STYLE&OPTIONS|FOOD&DRINKS| negative|
|  3|Nhìn chung dịch v...|      QUALITY|FOOD&DRINKS| negative|
|  3|Nhìn chung dịch v...|      GENERAL|    SERVICE| positive|
+---+--------------------+-------------+-----------+---------+
only showing top 5 rows


In [5]:
stopwords = set()

with open("stopwords.txt", "r", encoding="utf-8") as f:
    for line in f:
        stopwords.add(line.strip())

print(list(stopwords)[:10])  # test thử

['hôm qua', 'họ', 'anh', 'sau', 'sẽ', 'tôi', 'muốn', 'các', 'vẫn', 'giữa']


In [6]:
def preprocess(text):
    if text is None:
        return []

    # 1. lowercase
    text = text.lower()

    # 2. tách từ
    words = text.split()

    # 3. remove stopwords
    words = [w for w in words if w not in stopwords]

    return words

In [12]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

preprocess_udf = udf(preprocess, ArrayType(StringType()))

df = df.withColumn("processed", preprocess_udf(df["review"]))

df.select("review", "processed").show(5, truncate=False)

+---------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|review                                                                                       |processed                                                                                           |
+---------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+
|Vừa qua tôi có dùng dịch vụ tại Khách Sạn TTC Hotel Premium Ngọc Lan Ngọc Lan Đà Lạt.        |[vừa, qua, dùng, dịch, vụ, khách, sạn, ttc, hotel, premium, ngọc, lan, ngọc, lan, đà, lạt.]         |
|Tuy nhiên buffet sáng ở đây không được ngon và chưa đa dạng lắm.                             |[tuy, nhiên, buffet, sáng, đây, không, ngon, chưa, đa, dạng, lắm.]                                  |
|Tuy nhiên buff

In [13]:
# BÀI 2
from pyspark.sql.functions import explode, col

# tách từng từ thành từng dòng
word_df = df.select(explode(col("processed")).alias("word"))

# đếm tần số
word_count = word_df.groupBy("word") \
    .count() \
    .orderBy("count", ascending=False)

# top 5 từ xuất hiện nhiều nhất
top5_words = word_count.limit(5)

top5_words.show()

+-----+-----+
| word|count|
+-----+-----+
|phòng| 6588|
|khách| 4982|
|  sạn| 3928|
|   vụ| 3633|
|không| 3244|
+-----+-----+



In [14]:
category_stats = df.groupBy("category") \
    .count() \
    .orderBy("count", ascending=False)

category_stats.show()

+---------------+-----+
|       category|count|
+---------------+-----+
|        GENERAL| 5582|
|        COMFORT| 1373|
|    CLEANLINESS| 1364|
|        QUALITY| 1164|
|DESIGN&FEATURES|  939|
|         PRICES|  745|
|  MISCELLANEOUS|  335|
|  STYLE&OPTIONS|  267|
+---------------+-----+



In [15]:
aspect_stats = df.groupBy("aspect") \
    .count() \
    .orderBy("count", ascending=False)

aspect_stats.show()

+--------------+-----+
|        aspect|count|
+--------------+-----+
|         HOTEL| 3605|
|         ROOMS| 2501|
|       SERVICE| 2301|
|ROOM_AMENITIES| 1267|
|      LOCATION|  957|
|   FOOD&DRINKS|  696|
|    FACILITIES|  442|
+--------------+-----+



In [16]:
# Bai 3
from pyspark.sql.functions import col

aspect_sentiment = df.groupBy("aspect", "sentiment") \
    .count()

aspect_sentiment.show()

+--------------+---------+-----+
|        aspect|sentiment|count|
+--------------+---------+-----+
|       SERVICE|  neutral|  129|
|   FOOD&DRINKS| negative|  277|
|         HOTEL| positive| 2941|
|      LOCATION| positive|  843|
|      LOCATION| negative|   95|
|   FOOD&DRINKS| positive|  366|
|         HOTEL| negative|  341|
|         ROOMS| negative|  515|
|         HOTEL|  neutral|  323|
|    FACILITIES| positive|  213|
|    FACILITIES| negative|  211|
|       SERVICE| positive| 1906|
|ROOM_AMENITIES| positive|  834|
|       SERVICE| negative|  266|
|         ROOMS| positive| 1904|
|    FACILITIES|  neutral|   18|
|      LOCATION|  neutral|   19|
|ROOM_AMENITIES|  neutral|   30|
|         ROOMS|  neutral|   82|
|   FOOD&DRINKS|  neutral|   53|
+--------------+---------+-----+
only showing top 20 rows


In [17]:
negative_aspect = aspect_sentiment \
    .filter(col("sentiment") == "negative") \
    .orderBy("count", ascending=False)

negative_aspect.show(1)

+------+---------+-----+
|aspect|sentiment|count|
+------+---------+-----+
| ROOMS| negative|  515|
+------+---------+-----+
only showing top 1 row


In [18]:
positive_aspect = aspect_sentiment \
    .filter(col("sentiment") == "positive") \
    .orderBy("count", ascending=False)

positive_aspect.show(1)

+------+---------+-----+
|aspect|sentiment|count|
+------+---------+-----+
| HOTEL| positive| 2941|
+------+---------+-----+
only showing top 1 row


In [19]:
# Bai 4
from pyspark.sql.functions import explode, col

# tách từng từ nhưng vẫn giữ category + sentiment
word_df = df.select(
    "category",
    "sentiment",
    explode(col("processed")).alias("word")
)

word_df.show(5)

+--------+---------+----+
|category|sentiment|word|
+--------+---------+----+
| GENERAL| positive| vừa|
| GENERAL| positive| qua|
| GENERAL| positive|dùng|
| GENERAL| positive|dịch|
| GENERAL| positive|  vụ|
+--------+---------+----+
only showing top 5 rows


In [20]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# lọc positive
positive_words = word_df.filter(col("sentiment") == "positive")

# đếm theo category + word
pos_count = positive_words.groupBy("category", "word") \
    .count()

# tạo window theo category
windowSpec = Window.partitionBy("category").orderBy(desc("count"))

# lấy top 5 mỗi category
top5_positive = pos_count.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") <= 5)

top5_positive.show(truncate=False)

+---------------+-----+-----+----+
|category       |word |count|rank|
+---------------+-----+-----+----+
|CLEANLINESS    |phòng|1177 |1   |
|CLEANLINESS    |sạch |965  |2   |
|CLEANLINESS    |sẽ,  |600  |3   |
|CLEANLINESS    |tiện |530  |4   |
|CLEANLINESS    |ốc   |481  |5   |
|COMFORT        |hài  |530  |1   |
|COMFORT        |lòng |444  |2   |
|COMFORT        |phòng|433  |3   |
|COMFORT        |khách|427  |4   |
|COMFORT        |sạn  |357  |5   |
|DESIGN&FEATURES|phòng|528  |1   |
|DESIGN&FEATURES|không|227  |2   |
|DESIGN&FEATURES|sạch |218  |3   |
|DESIGN&FEATURES|khách|217  |4   |
|DESIGN&FEATURES|sạn  |185  |5   |
|GENERAL        |khách|2304 |1   |
|GENERAL        |phòng|2209 |2   |
|GENERAL        |vụ   |1933 |3   |
|GENERAL        |nhân |1848 |4   |
|GENERAL        |sạn  |1782 |5   |
+---------------+-----+-----+----+
only showing top 20 rows


In [21]:
# lọc negative
negative_words = word_df.filter(col("sentiment") == "negative")

# đếm theo category + word
neg_count = negative_words.groupBy("category", "word") \
    .count()

# dùng lại window
top5_negative = neg_count.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") <= 5)

top5_negative.show(truncate=False)

+---------------+-----+-----+----+
|category       |word |count|rank|
+---------------+-----+-----+----+
|CLEANLINESS    |phòng|193  |1   |
|CLEANLINESS    |không|170  |2   |
|CLEANLINESS    |sạch |84   |3   |
|CLEANLINESS    |sinh |74   |4   |
|CLEANLINESS    |vệ   |73   |5   |
|COMFORT        |phòng|203  |1   |
|COMFORT        |không|197  |2   |
|COMFORT        |khách|104  |3   |
|COMFORT        |sạn  |81   |4   |
|COMFORT        |hài  |70   |5   |
|DESIGN&FEATURES|phòng|238  |1   |
|DESIGN&FEATURES|không|160  |2   |
|DESIGN&FEATURES|khách|70   |3   |
|DESIGN&FEATURES|sạn  |58   |4   |
|DESIGN&FEATURES|cách |38   |5   |
|GENERAL        |không|369  |1   |
|GENERAL        |phòng|311  |2   |
|GENERAL        |khách|264  |3   |
|GENERAL        |sạn  |192  |4   |
|GENERAL        |nhân |149  |5   |
+---------------+-----+-----+----+
only showing top 20 rows


In [22]:
# Bài 5
from pyspark.sql.functions import explode, col

word_df = df.select(
    "category",
    explode(col("processed")).alias("word")
)

word_df.show(5)

+--------+----+
|category|word|
+--------+----+
| GENERAL| vừa|
| GENERAL| qua|
| GENERAL|dùng|
| GENERAL|dịch|
| GENERAL|  vụ|
+--------+----+
only showing top 5 rows


In [23]:
word_count = word_df.groupBy("category", "word") \
    .count()

word_count.show(5)

+---------------+-----+-----+
|       category| word|count|
+---------------+-----+-----+
|DESIGN&FEATURES|không|  398|
|        COMFORT|  ttc|   13|
|  MISCELLANEOUS|  bữa|    7|
|DESIGN&FEATURES|nghi.|   27|
|        COMFORT| mát.|   52|
+---------------+-----+-----+
only showing top 5 rows


In [24]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

windowSpec = Window.partitionBy("category").orderBy(desc("count"))

top5_related = word_count.withColumn(
    "rank", row_number().over(windowSpec)
).filter(col("rank") <= 5)

top5_related.show(truncate=False)

+---------------+-----+-----+----+
|category       |word |count|rank|
+---------------+-----+-----+----+
|CLEANLINESS    |phòng|1383 |1   |
|CLEANLINESS    |sạch |1052 |2   |
|CLEANLINESS    |sẽ,  |626  |3   |
|CLEANLINESS    |tiện |557  |4   |
|CLEANLINESS    |ốc   |537  |5   |
|COMFORT        |phòng|641  |1   |
|COMFORT        |hài  |612  |2   |
|COMFORT        |khách|536  |3   |
|COMFORT        |lòng |516  |4   |
|COMFORT        |không|504  |5   |
|DESIGN&FEATURES|phòng|777  |1   |
|DESIGN&FEATURES|không|398  |2   |
|DESIGN&FEATURES|khách|296  |3   |
|DESIGN&FEATURES|sạn  |250  |4   |
|DESIGN&FEATURES|sạch |246  |5   |
|GENERAL        |khách|2745 |1   |
|GENERAL        |phòng|2675 |2   |
|GENERAL        |vụ   |2191 |3   |
|GENERAL        |sạn  |2129 |4   |
|GENERAL        |nhân |2100 |5   |
+---------------+-----+-----+----+
only showing top 20 rows
